# EBNeRD Large Reranking - OPTIMIZED (GPU + Batching)

**Faster version with batch processing and vectorized operations**

Optimizations:
- Batch processing (500 impressions at a time) instead of row-by-row
- Load behaviors + history upfront for O(1) lookups
- Vectorized numpy operations for scoring
- Efficient parquet batching (flush every 100K scores)
- GPU for embeddings (sentence-transformers)

Expected: ~4-5 hours (vs 6-8 hours for row-by-row)

⚠️ Still the largest run — enable GPU and budget 5+ hours!

## Resume Workflow (if interrupted)

If this notebook times out mid-run:
1. Download `checkpoint.json`, `scores_parts/`, `predictions.txt`, `embeddings.npy`, `article_ids.npy` from this run's Output
2. Create a new Kaggle Dataset from those files (or upload as a new version)
3. In the **Parameters** cell below, set `RESUME_DIR = '/kaggle/input/<dataset-slug>/'`
4. Re-run this notebook — it will skip already-processed impressions and continue from the checkpoint

**Why**: `/kaggle/working/` is wiped at the start of each new notebook version, so checkpoints must be re-uploaded as input datasets to persist across versions.

## Setup

In [ ]:
import os
import sys
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from sentence_transformers import SentenceTransformer

sys.path.insert(0, '/kaggle/input/datasets/kspsvlnsiddardha/ebnerd-large-rerank-data/code')
from bm25_retrieval import InvertedIndex, tokenize
from fusion import weighted_fusion, scores_to_rank_permutation

print("✓ Imports successful")

## Parameters

In [ ]:
ALPHA = 0.5  # semantic weight; bm25 weight = 1 - ALPHA
BATCH_SIZE = 500  # Smaller batches for EBNeRD (larger articles)

DATA_DIR = Path('/kaggle/input/datasets/kspsvlnsiddardha/ebnerd-large-rerank-data')
SCORES_PATH = DATA_DIR / 'scores' / 'scores.parquet'
OUT_DIR = Path('/kaggle/working')
OUT_DIR.mkdir(exist_ok=True)

# Resume from a previous run's checkpoint dataset
RESUME_DIR = None  # Set to '/kaggle/input/<resume-dataset-slug>/' if resuming from a timeout
if RESUME_DIR:
    RESUME_DIR = Path(RESUME_DIR)

HAVE_CACHED_SCORES = SCORES_PATH.exists()

print(f"ALPHA: {ALPHA}")
print(f"BATCH_SIZE: {BATCH_SIZE}")
print(f"Using cached scores: {HAVE_CACHED_SCORES}")
# print(f"Resume dir: {RESUME_DIR if RESUME_DIR else 'None (fresh start)'}\")

In [ ]:
import json

# === CHECKPOINT LOADING ===
checkpoint_dir = RESUME_DIR if RESUME_DIR else OUT_DIR
checkpoint_file = checkpoint_dir / 'checkpoint.json'
pred_out_file = OUT_DIR / 'predictions.txt'
scores_parts_dir_old = checkpoint_dir / 'scores_parts'
scores_parts_dir_new = OUT_DIR / 'scores_parts'
embeddings_file = OUT_DIR / 'embeddings.npy'
article_ids_file = OUT_DIR / 'article_ids.npy'

checkpoint = {}
resume_from_index = 0
pred_lines_so_far = 0

if checkpoint_file.exists():
    with open(checkpoint_file) as f:
        checkpoint = json.load(f)
    resume_from_index = checkpoint.get('last_index', 0)
    pred_lines_so_far = checkpoint.get('pred_lines', 0)
    print(f"✓ RESUMING: {resume_from_index:,} impressions already processed, {pred_lines_so_far:,} predictions written")
else:
    print(f"✓ STARTING FRESH")

# Copy predictions.txt forward if cross-version resume
if RESUME_DIR and (RESUME_DIR / 'predictions.txt').exists() and not pred_out_file.exists():
    import shutil
    shutil.copy(RESUME_DIR / 'predictions.txt', pred_out_file)
    print(f"✓ Copied predictions.txt from checkpoint dir")

## Cache Check

In [ ]:
if HAVE_CACHED_SCORES:
    print(f"✓ Using cached scores")
else:
    print(f"✗ No cached scores - will build index + embeddings + score all candidates (budget 5+ hours)")

## Branch 1: Full Pipeline (OPTIMIZED with batching)

In [ ]:
if not HAVE_CACHED_SCORES:
    print("\n[1/6] Loading articles and building BM25 index...")
    articles_path = DATA_DIR / 'data' / 'articles.parquet'
    articles_df = pd.read_parquet(articles_path)
    
    article_text = {}
    for _, row in articles_df.iterrows():
        aid = row['article_id']
        title = str(row.get('title', '') or "")
        subtitle = str(row.get('subtitle', '') or "")
        text = title + " " + subtitle
        article_text[aid] = text.strip()
    
    index = InvertedIndex.build(articles_df[['article_id', 'title', 'subtitle']].copy())
    print(f"  ✓ BM25 index: {len(index.postings):,} terms, {index.N:,} documents")

In [ ]:
if not HAVE_CACHED_SCORES and not embeddings_file.exists():
    print("\n[2/6] Building semantic embeddings (GPU - 125.5K articles)...")
    model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-mpnet-base-v2')  # Danish-compatible multilingual model
    
    texts = []
    for _, row in articles_df.iterrows():
        title = str(row.get('title', '') or "")
        subtitle = str(row.get('subtitle', '') or "")
        text = title + " " + subtitle
        texts.append(text.strip() or "[empty]")
    
    # Batch encode with larger batch size for GPU
    embeddings = model.encode(texts, batch_size=128, show_progress_bar=True,
                              convert_to_numpy=True, normalize_embeddings=True)
    embeddings = embeddings.astype(np.float32)  # Save memory
    print(f"  ✓ Embeddings: {embeddings.shape}")
    
    # Save embeddings to disk for resume safety
    np.save(embeddings_file, embeddings)
    np.save(article_ids_file, articles_df['article_id'].values)
    print(f"  ✓ Saved embeddings cache to {embeddings_file}")
    
    aid_to_row = {aid: i for i, aid in enumerate(articles_df['article_id'])}
    
    del model  # Free GPU memory
elif embeddings_file.exists():
    print("\n[2/6] Loading cached embeddings...")
    embeddings = np.load(embeddings_file)
    aid_to_row = {aid: i for i, aid in enumerate(articles_df['article_id'])}
    print(f"  ✓ Loaded embeddings: {embeddings.shape}")

In [ ]:
if not HAVE_CACHED_SCORES:
    print("\n[3/6] Loading behaviors and history...")
    behaviors_path = DATA_DIR / 'data' / 'behaviors.parquet'
    history_path = DATA_DIR / 'data' / 'history.parquet'
    
    behaviors_df = pd.read_parquet(behaviors_path)
    history_df = pd.read_parquet(history_path)
    
    # Precompute user history map
    user_history_map = defaultdict(list)
    for _, row in history_df.iterrows():
        user_id = row['user_id']
        article_ids = row['article_id_fixed']
        if article_ids is None:
            article_ids = []
        elif isinstance(article_ids, np.ndarray):
            article_ids = article_ids.tolist()
        user_history_map[user_id] = article_ids
    
    print(f"  ✓ Loaded {len(behaviors_df):,} behaviors, {len(user_history_map):,} user histories")

In [ ]:
if not HAVE_CACHED_SCORES:
    print("\n[4/6] Batch processing 13.5M impressions with checkpointing...")
    
    # Prepare output directories
    scores_parts_dir_new.mkdir(exist_ok=True)
    pred_open_mode = 'a' if resume_from_index > 0 else 'w'
    
    # Determine next part file number
    existing_parts = list(scores_parts_dir_new.glob('scores_part_*.parquet'))
    if RESUME_DIR:
        existing_parts.extend(list(scores_parts_dir_old.glob('scores_part_*.parquet')))
    next_part_num = len(set(p.name for p in existing_parts))
    
    schema = pa.schema([
        ('impression_id', pa.int64()),
        ('article_id', pa.int64()),
        ('bm25_score', pa.float32()),
        ('semantic_score', pa.float32())
    ])
    
    pred_file = open(pred_out_file, pred_open_mode)
    score_rows = []
    pred_lines = pred_lines_so_far
    part_num = next_part_num
    
    # Process in batches with resume skip-ahead
    for batch_start in range(0, len(behaviors_df), BATCH_SIZE):
        batch_end = min(batch_start + BATCH_SIZE, len(behaviors_df))
        batch_data = behaviors_df.iloc[batch_start:batch_end]
        
        # Skip already-processed impressions on resume
        if batch_start < resume_from_index < batch_end:
            skip_idx = resume_from_index - batch_start
            batch_data = batch_data.iloc[skip_idx:]
        elif batch_start < resume_from_index:
            continue
        
        for _, row in batch_data.iterrows():
            impression_id = row['impression_id']
            user_id = row['user_id']
            article_ids_inview = row['article_ids_inview']
            
            if article_ids_inview is None:
                article_ids_inview = []
            elif isinstance(article_ids_inview, np.ndarray):
                article_ids_inview = article_ids_inview.tolist()
            elif not isinstance(article_ids_inview, list):
                article_ids_inview = list(article_ids_inview) if hasattr(article_ids_inview, '__iter__') else []
            
            if not article_ids_inview:
                continue
            
            history_ids = user_history_map.get(user_id, [])
            
            # BM25 scoring
            bm25_scores = {}
            if history_ids:
                query_text = " ".join(article_text.get(aid, "") for aid in history_ids)
                qtf = Counter(tokenize(query_text))
                if qtf:
                    bm25_scores = index.score_documents_batch(set(article_ids_inview), qtf, k1=1.5, b=0.75)
            
            for aid in article_ids_inview:
                if aid not in bm25_scores:
                    bm25_scores[aid] = 0.0
            
            # Semantic scoring (vectorized)
            semantic_scores = {}
            if history_ids:
                history_embs = []
                for aid in history_ids:
                    if aid in aid_to_row:
                        history_embs.append(embeddings[aid_to_row[aid]])
                
                if history_embs:
                    query_emb = np.mean(history_embs, axis=0)
                else:
                    query_emb = np.zeros(embeddings.shape[1], dtype=np.float32)
            else:
                query_emb = np.zeros(embeddings.shape[1], dtype=np.float32)
            
            query_norm = np.linalg.norm(query_emb)
            if query_norm > 1e-8:
                query_normalized = query_emb / query_norm
                for aid in article_ids_inview:
                    if aid in aid_to_row:
                        cand_vec = embeddings[aid_to_row[aid]]
                        semantic_scores[aid] = float(np.dot(query_normalized, cand_vec))
                    else:
                        semantic_scores[aid] = 0.0
            else:
                semantic_scores = {aid: 0.0 for aid in article_ids_inview}
            
            # Append to batch
            for aid in article_ids_inview:
                score_rows.append({
                    'impression_id': int(impression_id),
                    'article_id': int(aid),
                    'bm25_score': bm25_scores.get(aid, 0.0),
                    'semantic_score': semantic_scores.get(aid, 0.0)
                })
            
            # Fuse and write predictions
            retriever_scores = {"bm25": bm25_scores, "semantic": semantic_scores}
            fused = weighted_fusion(article_ids_inview, retriever_scores, {"bm25": 1 - ALPHA, "semantic": ALPHA})
            ranks = scores_to_rank_permutation(article_ids_inview, fused)
            pred_file.write(f"{impression_id} [{','.join(str(r) for r in ranks)}]\n")
            pred_lines += 1
        
        # Flush scores to part file every 100k rows
        if len(score_rows) >= 100000:
            part_path = scores_parts_dir_new / f'scores_part_{part_num:04d}.parquet'
            table = pa.Table.from_pylist(score_rows, schema=schema)
            pq.write_table(table, str(part_path))
            score_rows = []
            part_num += 1
            
            # Save checkpoint
            checkpoint_data = {'last_index': batch_end, 'pred_lines': pred_lines}
            with open(str(checkpoint_dir / 'checkpoint.json'), 'w') as f:
                json.dump(checkpoint_data, f)
        
        if batch_end % 100000 <= BATCH_SIZE:
            print(f"  Processed {batch_end:,} impressions, {pred_lines:,} predictions, {part_num} part files")
    
    # Final flush
    if score_rows:
        part_path = scores_parts_dir_new / f'scores_part_{part_num:04d}.parquet'
        table = pa.Table.from_pylist(score_rows, schema=schema)
        pq.write_table(table, str(part_path))
        
        checkpoint_data = {'last_index': len(behaviors_df), 'pred_lines': pred_lines}
        with open(str(checkpoint_dir / 'checkpoint.json'), 'w') as f:
            json.dump(checkpoint_data, f)
    
    pred_file.close()
    
    print(f"\n✓ Wrote {pred_lines:,} predictions to {pred_out_file}")
    print(f"✓ Wrote {part_num + 1} score part files to {scores_parts_dir_new}")

## Verify and Zip

In [ ]:
print("\nVerification:")
pred_out = OUT_DIR / 'predictions.txt'
if pred_out.exists():
    with open(pred_out) as f:
        n_lines = sum(1 for _ in f)
    print(f"✓ predictions.txt: {n_lines:,} lines (expect 13,536,710)")
    
    if n_lines == 13536710:
        print("  ✓✓✓ COMPLETE!")
    else:
        print(f"  ⚠ INCOMPLETE ({n_lines / 13536710 * 100:.1f}%)")
else:
    print(f"✗ predictions.txt not found")

# Combine scores from both old and new part files (if cross-version resume)
print("\nCombining score part files...")
score_parts_all = list(scores_parts_dir_new.glob('scores_part_*.parquet'))
if RESUME_DIR:
    score_parts_all.extend(list(scores_parts_dir_old.glob('scores_part_*.parquet')))

if score_parts_all:
    print(f"  Found {len(score_parts_all)} part files total")
    all_scores = pd.concat([pd.read_parquet(p) for p in sorted(score_parts_all)], ignore_index=True)
    print(f"  Combined into {len(all_scores):,} score rows")
    print(f"  Unique impressions: {all_scores['impression_id'].nunique():,}")

import zipfile
print("\nZipping...")
zip_out = OUT_DIR / 'predictions.zip'
with zipfile.ZipFile(zip_out, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(pred_out, arcname='predictions.txt')
print(f"✓ Created {zip_out}")
print(f"\n✓✓✓ Done!")